# does wikipedia attention predict stock volatility?

the idea is pretty simple. before a stock goes wild, do people start reading about the company? i pull daily wikipedia pageviews for ~25 big US companies, pull their stock prices, and check if attention tells us anything about next week's volatility that we couldn't already get from past volatility and trading volume.

the boring way to predict vol is the HAR model (yesterday's vol, last week's vol, last month's vol). that's my baseline. if wikipedia can't beat that, then it's just noise and i say so honestly.

**what's in here**
1. setup
2. get the data (prices + wikipedia views), cached into `data/`
3. have a look at it
4. build features + target
5. who leads who? (does attention come before vol or after)
6. walk forward models
7. results
8. does a surge in attention actually matter
9. notes / things to write up

## 1. setup

if you're missing packages run the first line (uncomment it). if the widgets don't show up in vs code, make sure `ipywidgets` is installed and reload the window.

In [ ]:
# %pip install yfinance plotly ipywidgets scikit-learn pandas numpy requests

import time
import warnings
from datetime import date
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

START = "2017-01-01"
END = date.today().isoformat()

## 2. data

two sources, both free, no api keys:
- **prices**: daily OHLCV from yahoo finance via `yfinance`
- **attention**: daily pageviews of each company's english wikipedia article from the wikimedia REST api (human users only, all platforms)

the ticker to article mapping is hand written below. wikipedia article titles have to match exactly (and be the real article, not a redirect, otherwise the views are tiny). if one fails to download the code just prints it and moves on, so fix the title and delete `data/wiki_views.csv` to refetch.

everything gets cached into `data/` so you only hit the apis once. commit that folder to github so the repo actually has the data in it.

In [ ]:
UNIVERSE = {
    "AAPL": "Apple_Inc.",
    "MSFT": "Microsoft",
    "AMZN": "Amazon_(company)",
    "GOOGL": "Alphabet_Inc.",
    "META": "Meta_Platforms",
    "TSLA": "Tesla,_Inc.",
    "NVDA": "Nvidia",
    "NFLX": "Netflix",
    "AMD": "Advanced_Micro_Devices",
    "INTC": "Intel",
    "ORCL": "Oracle_Corporation",
    "IBM": "IBM",
    "JPM": "JPMorgan_Chase",
    "GS": "Goldman_Sachs",
    "BAC": "Bank_of_America",
    "C": "Citigroup",
    "XOM": "ExxonMobil",
    "CVX": "Chevron_Corporation",
    "BA": "Boeing",
    "F": "Ford_Motor_Company",
    "DIS": "The_Walt_Disney_Company",
    "KO": "The_Coca-Cola_Company",
    "MCD": "McDonald's",
    "NKE": "Nike,_Inc.",
    "PFE": "Pfizer",
    "WMT": "Walmart",
}

pd.DataFrame({"ticker": list(UNIVERSE), "wiki_article": list(UNIVERSE.values())}).to_csv(
    DATA_DIR / "universe.csv", index=False
)

In [ ]:
PRICE_FILE = DATA_DIR / "prices.csv"

def download_prices():
    raw = yf.download(list(UNIVERSE), start=START, end=END, auto_adjust=True,
                      progress=False, group_by="ticker")
    frames = []
    for t in UNIVERSE:
        try:
            df = raw[t][["Open", "High", "Low", "Close", "Volume"]].dropna().copy()
        except KeyError:
            print("no price data for", t)
            continue
        df["ticker"] = t
        df.index.name = "date"
        frames.append(df.reset_index())
    return pd.concat(frames, ignore_index=True)

if PRICE_FILE.exists():
    prices = pd.read_csv(PRICE_FILE, parse_dates=["date"])
else:
    prices = download_prices()
    prices.to_csv(PRICE_FILE, index=False)

print(prices.shape, prices.ticker.nunique(), "tickers")
prices.head()

In [ ]:
WIKI_FILE = DATA_DIR / "wiki_views.csv"
# wikimedia asks you to send a real user agent, put your own email in here
HEADERS = {"User-Agent": "wiki-vol-student-project/0.1 (your-email@example.com)"}

def get_views(article, start="20170101", end=None):
    end = end or date.today().strftime("%Y%m%d")
    url = (
        "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"en.wikipedia/all-access/user/{quote(article, safe='')}/daily/{start}/{end}"
    )
    r = requests.get(url, headers=HEADERS, timeout=30)
    if r.status_code != 200:
        print("failed:", article, r.status_code)
        return None
    df = pd.DataFrame(r.json()["items"])[["timestamp", "views"]]
    df["date"] = pd.to_datetime(df["timestamp"].str[:8], format="%Y%m%d")
    return df[["date", "views"]]

if WIKI_FILE.exists():
    wiki = pd.read_csv(WIKI_FILE, parse_dates=["date"])
else:
    parts = []
    for tkr, art in UNIVERSE.items():
        v = get_views(art)
        if v is not None:
            v["ticker"] = tkr
            parts.append(v)
        time.sleep(0.3)  # be nice to the api
    wiki = pd.concat(parts, ignore_index=True)
    wiki.to_csv(WIKI_FILE, index=False)

print(wiki.shape, wiki.ticker.nunique(), "tickers")
wiki.head()

In [ ]:
# quick coverage check, make sure nothing is missing
cov_p = prices.groupby("ticker").date.agg(price_start="min", price_end="max", price_rows="count")
cov_w = wiki.groupby("ticker").date.agg(wiki_start="min", wiki_end="max", wiki_rows="count")
cov = cov_p.join(cov_w, how="outer")
cov

## 3. have a look

first the thing we're predicting. i use the **parkinson estimator** for daily variance, which uses the high and low of the day instead of just the close, so it's way less noisy than squared returns:

`pv = ln(High / Low)^2 / (4 ln 2)`

pick a ticker below. top panel is price, middle is rolling vol (annualised %), bottom is wikipedia views on a log scale. zoom with the bottom slider. look at where the spikes line up.

In [ ]:
def park_var(g):
    return (np.log(g["High"] / g["Low"]) ** 2) / (4 * np.log(2))

available = sorted(set(prices.ticker.unique()) & set(wiki.ticker.unique()))

def show_ticker(ticker="NVDA", window=22):
    g = prices[prices.ticker == ticker].sort_values("date").set_index("date")
    w = wiki[wiki.ticker == ticker].set_index("date")["views"]
    vol = np.sqrt(park_var(g).rolling(window).mean() * 252) * 100

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                        row_heights=[0.35, 0.3, 0.35],
                        subplot_titles=("price", f"{window}d rolling vol (annualised %)", "wikipedia views / day"))
    fig.add_trace(go.Scatter(x=g.index, y=g["Close"], name="close", line=dict(width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=vol.index, y=vol, name="vol", line=dict(width=1, color="crimson")), row=2, col=1)
    fig.add_trace(go.Scatter(x=w.index, y=w, name="views", line=dict(width=1, color="seagreen")), row=3, col=1)
    fig.update_yaxes(type="log", row=3, col=1)
    fig.update_xaxes(rangeslider=dict(visible=True, thickness=0.05), row=3, col=1)
    fig.update_layout(height=700, showlegend=False, title=ticker, margin=dict(t=60, b=20))
    fig.show()

widgets.interact(
    show_ticker,
    ticker=widgets.Dropdown(options=available, value="NVDA" if "NVDA" in available else available[0]),
    window=widgets.IntSlider(value=22, min=5, max=66, step=1),
);

## 4. features and target

**target**: log of the average parkinson variance over the next 5 trading days (t+1 to t+5).

**baseline features (HAR)**
- `har_d`: log of today's variance
- `har_w`: log of the last 5 day average variance
- `har_m`: log of the last 22 day average variance

**controls**
- `ret5`: 5 day return (down moves tend to be followed by more vol)
- `vol_surge`: trading volume now vs its 60 day norm. this one matters, because attention could just be a proxy for volume, and i want to know if wikipedia adds something beyond that

**the unusual data**
- `wiki_surge`: log views, 7 day average minus 90 day average (is this company getting more attention than usual)
- `wiki_1d`: today's log views minus the 7 day average (a one day spike)

wiki views are daily including weekends, so i compute the rolling stuff on the calendar series first and then line it up with trading days. nothing here looks forward, every feature only uses data up to and including day t.

In [ ]:
EPS = 1e-6

def build_panel(prices, wiki):
    out = []
    for t, g in prices.groupby("ticker"):
        w = wiki[wiki.ticker == t]
        if w.empty:
            continue
        g = g.sort_values("date").set_index("date")
        w = w.set_index("date")["views"].astype(float)
        w = w[~w.index.duplicated()].asfreq("D").interpolate(limit=3)
        lw = np.log1p(w)

        wf = pd.DataFrame({
            "wiki_surge": lw.rolling(7, min_periods=5).mean() - lw.rolling(90, min_periods=60).mean(),
            "wiki_1d": lw - lw.rolling(7, min_periods=5).mean(),
        }).reindex(g.index)

        pv = park_var(g).clip(lower=EPS)
        lv = np.log(g["Volume"].replace(0, np.nan))

        f = pd.DataFrame(index=g.index)
        f["ticker"] = t
        f["har_d"] = np.log(pv)
        f["har_w"] = np.log(pv.rolling(5).mean())
        f["har_m"] = np.log(pv.rolling(22).mean())
        f["ret5"] = g["Close"].pct_change(5)
        f["vol_surge"] = lv.rolling(5).mean() - lv.rolling(60).mean()
        f["wiki_surge"] = wf["wiki_surge"]
        f["wiki_1d"] = wf["wiki_1d"]
        # how surprising was today's vol vs its own recent norm (used in the lead lag chart)
        f["lp_surprise"] = np.log(pv) - np.log(pv).rolling(60).mean()
        # target: mean variance over t+1..t+5
        fut = pv.rolling(5).mean().shift(-5)
        f["fut_pv"] = fut
        f["y"] = np.log(fut)
        out.append(f)

    panel = pd.concat(out).reset_index().rename(columns={"index": "date"})
    return panel.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

panel = build_panel(prices, wiki)
print(panel.shape, panel.date.min().date(), "to", panel.date.max().date())
panel.describe().T.round(3)

In [ ]:
# how do the features line up with the target (plain correlations, nothing fancy)
feat_cols = ["har_d", "har_w", "har_m", "ret5", "vol_surge", "wiki_surge", "wiki_1d"]
corr = panel[feat_cols + ["y"]].corr()["y"].drop("y").sort_values()

fig = px.bar(corr, orientation="h", title="correlation of each feature with next-week log variance")
fig.update_layout(showlegend=False, xaxis_title="correlation", yaxis_title="", height=380)
fig.show()

## 5. who leads who?

before any modelling, a sanity check on direction. for each lag k i correlate today's `wiki_surge` with how surprising the vol was k days away. positive k means vol happens after the attention (attention leads), negative k means vol was first and the attention came after (attention reacts).

if the peak is at zero or on the negative side, wikipedia is mostly just reacting to the news, which is what you'd kind of expect, and the predictive power will be small.

**TODO (after running):** say what the shape looks like and where the peak is.

In [ ]:
ks = list(range(-10, 11))
rows = []
for t, g in panel.groupby("ticker"):
    g = g.sort_values("date")
    for k in ks:
        rows.append((t, k, g["wiki_surge"].corr(g["lp_surprise"].shift(-k))))
ll = pd.DataFrame(rows, columns=["ticker", "k", "corr"])
ll_mean = ll.groupby("k")["corr"].mean()

fig = go.Figure()
for t, g in ll.groupby("ticker"):
    fig.add_trace(go.Scatter(x=g.k, y=g["corr"], mode="lines", line=dict(width=1, color="lightgrey"),
                             name=t, hoverinfo="name+y", showlegend=False))
fig.add_trace(go.Scatter(x=ll_mean.index, y=ll_mean.values, mode="lines+markers",
                         line=dict(width=3, color="crimson"), name="average"))
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(title="corr(wiki surge today, vol surprise k days later)",
                  xaxis_title="k (days). negative = vol came first", yaxis_title="correlation", height=420)
fig.show()

## 6. walk forward models

no random train/test split here, that would leak the future into the past (vol clusters, so neighbouring days look the same). instead for each test year i train on everything before it and predict that year. there's also a 7 day gap between train and test because the target looks 5 days ahead.

models:
- **HAR**: baseline, 3 features
- **HAR + controls**: adds 5d return and volume surge
- **HAR + controls + wiki**: adds the two wikipedia features
- **GBM**: gradient boosting on everything, in case there's some nonlinear stuff going on

all of them are pooled over all the stocks, to keep it simple. (a thing to try later: per stock models.)

In [ ]:
HAR = ["har_d", "har_w", "har_m"]
CTRL = HAR + ["ret5", "vol_surge"]
WIKI = CTRL + ["wiki_surge", "wiki_1d"]
FEATURE_SETS = {"HAR": HAR, "HAR + controls": CTRL, "HAR + controls + wiki": WIKI}
GAP_DAYS = 7

first_test_year = 2021
test_years = list(range(first_test_year, panel.date.dt.year.max() + 1))

pred_parts = []
for yr in test_years:
    test = panel[panel.date.dt.year == yr]
    if test.empty:
        continue
    cutoff = test.date.min() - pd.Timedelta(days=GAP_DAYS)
    train = panel[panel.date < cutoff]

    out = test[["date", "ticker", "y", "fut_pv", "wiki_surge", "wiki_1d"]].copy()
    out["year"] = yr
    for name, cols in FEATURE_SETS.items():
        m = LinearRegression().fit(train[cols], train["y"])
        out[name] = m.predict(test[cols])

    gbm = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=300, random_state=0)
    gbm.fit(train[WIKI], train["y"])
    out["GBM"] = gbm.predict(test[WIKI])
    pred_parts.append(out)

oos = pd.concat(pred_parts, ignore_index=True)
MODEL_NAMES = list(FEATURE_SETS) + ["GBM"]
print(oos.shape, "out of sample rows")

## 7. results

three scores:
- **MSE** on the log variance (lower is better)
- **R2 vs HAR**: 1 minus (model MSE / HAR MSE). positive means it beats the baseline, negative means it's worse than the dumb baseline
- **QLIKE**: the usual loss for variance forecasts (lower is better). note i use exp(prediction) as the variance forecast, which is slightly biased low, but it's the same for every model so the comparison is fair

heads up on stats: the target overlaps (the 5 day windows for consecutive days share 4 days), so errors are heavily autocorrelated and a naive t-stat would be way too optimistic. i'm only comparing scores here, not claiming significance.

In [ ]:
def qlike(actual_var, pred_log):
    r = actual_var / np.exp(pred_log)
    return r - np.log(r) - 1

def scores(df):
    base_mse = ((df["y"] - df["HAR"]) ** 2).mean()
    rows = []
    for m in MODEL_NAMES:
        mse = ((df["y"] - df[m]) ** 2).mean()
        rows.append({"model": m, "MSE (log)": mse,
                     "R2 vs HAR": 1 - mse / base_mse,
                     "QLIKE": qlike(df["fut_pv"], df[m]).mean()})
    return pd.DataFrame(rows).set_index("model")

overall = scores(oos)
overall.round(4)

In [ ]:
# R2 vs HAR by year, so you can see if the wiki thing is stable or just one lucky year
by_year = []
for yr, d in oos.groupby("year"):
    s = scores(d)["R2 vs HAR"].drop("HAR").reset_index()
    s["year"] = yr
    by_year.append(s)
by_year = pd.concat(by_year)

fig = px.bar(by_year, x="year", y="R2 vs HAR", color="model", barmode="group",
             title="out of sample R2 relative to the HAR baseline, by year")
fig.add_hline(y=0, line_dash="dot")
fig.update_layout(height=420, yaxis_tickformat=".1%")
fig.show()

In [ ]:
# same thing but split by stock, for the wiki model vs the controls-only model
rows = []
for t, d in oos.groupby("ticker"):
    mse_ctrl = ((d["y"] - d["HAR + controls"]) ** 2).mean()
    mse_wiki = ((d["y"] - d["HAR + controls + wiki"]) ** 2).mean()
    rows.append({"ticker": t, "gain from adding wiki": 1 - mse_wiki / mse_ctrl})
per_t = pd.DataFrame(rows).sort_values("gain from adding wiki")

fig = px.bar(per_t, x="gain from adding wiki", y="ticker", orientation="h",
             title="does wiki help for this stock? (MSE improvement over HAR + controls)")
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(height=650, xaxis_tickformat=".1%")
fig.show()

In [ ]:
# forecasts vs reality for one stock, pick any. shown as annualised vol %
def to_vol(logvar):
    return np.sqrt(np.exp(logvar) * 252) * 100

def show_forecast(ticker="TSLA"):
    d = oos[oos.ticker == ticker].sort_values("date")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d.date, y=to_vol(d["y"]), name="actual next 5d vol", line=dict(width=1.5, color="black")))
    fig.add_trace(go.Scatter(x=d.date, y=to_vol(d["HAR"]), name="HAR", line=dict(width=1, color="royalblue")))
    fig.add_trace(go.Scatter(x=d.date, y=to_vol(d["HAR + controls + wiki"]), name="HAR + controls + wiki",
                             line=dict(width=1, color="crimson")))
    fig.update_layout(title=f"{ticker}: out of sample vol forecasts", yaxis_title="annualised vol %",
                      height=450, xaxis=dict(rangeslider=dict(visible=True, thickness=0.05)))
    fig.show()

avail_oos = sorted(oos.ticker.unique())
widgets.interact(show_forecast, ticker=widgets.Dropdown(options=avail_oos,
                 value="TSLA" if "TSLA" in avail_oos else avail_oos[0]));

**TODO (after running):** write up the results table here. things worth saying:
- does `HAR + controls` beat plain HAR? (it usually does, volume is informative)
- does adding wiki beat `HAR + controls`, and by how much? is it consistent across years or just one year?
- does the GBM do anything extra or does it just overfit?
- which stocks gain the most from wiki, and is there a pattern (meme-ish stocks like TSLA / NVDA vs boring ones like KO / WMT)?

## 8. does a surge in attention actually matter?

another way to look at it. take the forecast errors from the model *without* wikipedia, and bucket the days by how big the wiki surge was. if attention carries real extra info, the days with the biggest surge should have positive errors (vol came in higher than the model without wiki expected).

error bars are naive standard errors so treat them as optimistic (overlapping windows again).

In [ ]:
oos["resid_no_wiki"] = oos["y"] - oos["HAR + controls"]
oos["surge_q"] = pd.qcut(oos["wiki_surge"], 5, labels=["Q1 (lowest)", "Q2", "Q3", "Q4", "Q5 (biggest surge)"])

q = oos.groupby("surge_q", observed=True)["resid_no_wiki"].agg(["mean", "std", "count"])
q["se"] = q["std"] / np.sqrt(q["count"])

fig = go.Figure(go.Bar(x=q.index.astype(str), y=q["mean"], error_y=dict(type="data", array=q["se"] * 1.96),
                       marker_color="seagreen"))
fig.add_hline(y=0, line_dash="dot")
fig.update_layout(title="avg forecast error of the no-wiki model, by wiki surge quintile",
                  yaxis_title="actual minus predicted log variance", height=420)
fig.show()
q.round(4)

In [ ]:
# what the final linear model puts weight on (standardised so the sizes are comparable)
sc = StandardScaler().fit(panel[WIKI])
final = LinearRegression().fit(sc.transform(panel[WIKI]), panel["y"])
coefs = pd.Series(final.coef_, index=WIKI).sort_values()

fig = px.bar(coefs, orientation="h", title="standardised coefficients, HAR + controls + wiki (fit on all data)")
fig.update_layout(showlegend=False, height=380, xaxis_title="effect on next-week log variance per 1 sd", yaxis_title="")
fig.show()

## 9. notes / write up

**TODO (after running):** fill this in once you've seen the numbers. a rough structure:

- **what i found**: one or two sentences, with the actual R2 numbers
- **why it might be that way**: attention mostly reacts to news rather than predicting it (see the lead lag chart). a big caveat is that earnings dates are known in advance, so a lot of attention spikes are just scheduled events, and the vol around them was already predictable
- **limitations**: 26 mega caps only (survivorship, these are the companies that survived and got huge), pooled models, overlapping targets, no transaction costs or anything because this is forecasting not trading
- **next things to try**
  - add earnings dates as a control (yfinance can give them, a bit flaky) and see if wiki still adds anything
  - per stock models instead of pooled
  - try google trends or reddit mentions and compare with wikipedia
  - use intraday data for a cleaner realised vol target
  - turn it into a tiny strategy: sort stocks by forecast vol change and see if a vol-timed portfolio does better